In [1]:
import os 
import sys

In [9]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [10]:
print(fo.list_datasets())

['FLPLAN', 'UM_teste', 'WPonly_1024ov224', 'flplan200', 'flplanpatches', 'nc765', 'nc765_T1024ov224', 'nc765_T2048ov224', 'tiles2048_merged', 'tiles_merged', 'tiles_merged_2048', 'train_nc_wp_T1024ov224', 'umflplan_T1024ov224', 'umflplan_T2048ov224', 'wp125', 'wp125_T1024ov224', 'wp125_T2048ov224']


In [4]:
dataset = fo.load_dataset("WPonly_1024ov224")
dataset

Name:        WPonly_1024ov224
Media type:  image
Num samples: 1455
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    type_label:       fiftyone.core.fields.StringField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    region:           fiftyone.core.fields.StringField
    mission_name:     fiftyone.core.fields.StringField
    parent_name:      fiftyone.core.fields.StringField
    gt_field:         fiftyone.core.fields.StringField
    min_area_ratio:   fiftyone.core.fields.FloatField
    n_boxes_in_tile:  fiftyone.core.field

In [6]:
dataset.count_values("parent_name")

{'FRIWEN_M19': 45,
 'FRIWEN_M8': 315,
 'GAM_M10': 60,
 'GAM_M1': 180,
 'FRIWEN_M11': 360,
 'FRIWEN_M12': 165,
 'FRIWEN_M53': 60,
 'MANTASANDY_M5': 105,
 'GAM_M2': 165}

In [7]:
import random
from sklearn.model_selection import train_test_split
from fiftyone import ViewField as F


def tag_train_val_seeded_split(
    dataset,
    stratify_by: str   = "m_flight",
    val_size:    float = 0.15,
    num_seeds:   int   = 1,
    verbose:     bool  = True,
):
    """
    Splits a FiftyOne dataset into train / val by tagging samples,
    stratified by a categorical field (e.g. flight mission).

    Source-domain (NC) version: NO test split. The entire dataset is
    divided into train / val only. Fine-tuning and testing happen later
    on the target domain.

    Parameters
    ----------
    dataset     : FiftyOne dataset
    stratify_by : sample field used for stratification (e.g. "m_flight")
    val_size    : fraction of samples used for validation
    num_seeds   : number of independent splits to generate
    verbose     : print progress

    Tags written
    ------------
    train_{seed}, val_{seed}
    """

    def _log(msg):
        if verbose:
            print(f"  {msg}")

    total_images = dataset.count()
    flight_counts = dataset.count_values(stratify_by)

    _log(f"Dataset: {total_images} images  |  "
         f"{len(flight_counts)} flights via '{stratify_by}'")
    _log(f"Target split — train:{1 - val_size:.0%}  val:{val_size:.0%}")

    all_ids    = dataset.values("id")
    all_strata = dataset.values(stratify_by)

    for seed in range(num_seeds):
        _log(f"\n── Seed {seed} ──────────────────────────────")

        train_ids, val_ids = train_test_split(
            all_ids,
            test_size=val_size,
            stratify=all_strata,
            random_state=seed,
            shuffle=True,
        )

        dataset.select(train_ids).tag_samples(f"train_{seed}")
        dataset.select(val_ids).tag_samples(f"val_{seed}")

        _log(f"Tagged: train_{seed} ({len(train_ids)})  "
             f"val_{seed} ({len(val_ids)})")

        if verbose:
            print(f"\n  Split summary for seed {seed}:")
            print(f"    train_{seed} : {len(train_ids):>5}  "
                  f"({len(train_ids)/total_images:.1%})")
            print(f"    val_{seed}   : {len(val_ids):>5}  "
                  f"({len(val_ids)/total_images:.1%})")

    print(f"\nDone. {num_seeds} seed(s) tagged.")

In [8]:
tag_train_val_seeded_split(
    dataset,
    stratify_by="parent_name",
    val_size= 0.2
)

  Dataset: 1455 images  |  9 flights via 'parent_name'
  Target split — train:80%  val:20%
  
── Seed 0 ──────────────────────────────
  Tagged: train_0 (1164)  val_0 (291)

  Split summary for seed 0:
    train_0 :  1164  (80.0%)
    val_0   :   291  (20.0%)

Done. 1 seed(s) tagged.


In [11]:
wponly = fo.load_dataset("WPonly_1024ov224")
nc_dataset = fo.load_dataset("nc765_T1024ov224")

In [15]:
wponly.count_values("tags")

{'val_0': 291, 'train_0': 1164}

In [17]:
nc_dataset.count_values("tags")

{'val_0': 860, 'train_0': 4868}

In [22]:
val_wponly_filepath = wponly.match_tags(
    "val_0"
).values("filepath")
train_wponly_filepath = wponly.match_tags(
    "train_0"
).values("filepath")
val_nc_filepath = nc_dataset.match_tags(
    "val_0"
).values("filepath")
train_nc_filepath = nc_dataset.match_tags(
    "train_0"
).values("filepath")


In [37]:
print(
    len(val_wponly_filepath),
    len(train_wponly_filepath),
    len(val_nc_filepath),
    len(train_nc_filepath)
    )

291 1164 860 4868


In [20]:
train_ncwp_dataset = fo.load_dataset(
    "train_nc_wp_T1024ov224"
)


In [44]:
train_ncwp_dataset.untag_samples("train_0")

In [55]:
ids = train_ncwp_dataset.select_by("filepath",train_nc_filepath).values("id")

In [56]:
train_ncwp_dataset[ids].tag_samples("train_0")

In [57]:
train_ncwp_dataset.count_values("tags")

{'train_0': 6032, 'val_0': 1151}

In [34]:
train_ncwp_dataset

Name:        train_nc_wp_T1024ov224
Media type:  image
Num samples: 7183
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    type_label:       fiftyone.core.fields.StringField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    region:           fiftyone.core.fields.StringField
    mission_name:     fiftyone.core.fields.StringField
    parent_name:      fiftyone.core.fields.StringField
    gt_field:         fiftyone.core.fields.StringField
    min_area_ratio:   fiftyone.core.fields.FloatField
    n_boxes_in_tile:  fiftyone.core